In [ ]:
import os
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image

# Set the model and device
model_id = r"E:\Snapfolia - CS\grounding-dino-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize processor and model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

# Set the root directory containing the classes and images
root_dir = "F:\SNAP-DS-4"

# Text to detect (leaves in this case)
text = "a leaf. a leaves."

# Create the output directory for the extracted data
output_root_dir = "F:\SNAP-DS-4-EXTRACTED"
os.makedirs(output_root_dir, exist_ok=True)

# Initialize class index mapping
class_names = os.listdir(root_dir)
class_name_to_index = {class_name: idx for idx, class_name in enumerate(class_names)}

# Iterate over all classes in the root directory
for class_name, class_idx in class_name_to_index.items():
    class_folder = os.path.join(root_dir, class_name)
    
    if os.path.isdir(class_folder):
        print()
        print("--------------------------------------------------------")
        print(f"Processing class: {class_name} (Class {class_idx})")
        
        # Create a directory to save results for each class (use lowercase for class names)
        class_output_dir = os.path.join(output_root_dir, f"class{class_idx}")
        os.makedirs(class_output_dir, exist_ok=True)
        
        # Create subdirectories for images and labels
        images_dir = os.path.join(class_output_dir, "images")
        labels_dir = os.path.join(class_output_dir, "labels")
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
        
        # Iterate over all images in the class folder
        for image_name in os.listdir(class_folder):
            image_path = os.path.join(class_folder, image_name)
            
            if image_name.endswith(('.jpg', '.jpeg', '.png')):
                print(f"->Processing image: {image_name}")
                
                # Open the image
                image = Image.open(image_path)
                image_width, image_height = image.size

                # Prepare the inputs for the model
                inputs = processor(images=image, text=text, return_tensors="pt").to(device)
                
                # Perform inference
                with torch.no_grad():
                    outputs = model(**inputs)

                # Post-process the results
                results = processor.post_process_grounded_object_detection(
                    outputs,
                    inputs.input_ids,
                    box_threshold=0.3,
                    text_threshold=0.3,
                    target_sizes=[image.size[::-1]]
                )

                # Create the output file for labels
                output_file = os.path.join(labels_dir, f"{os.path.splitext(image_name)[0]}.txt")

                # Check if any detections were made
                if len(results) > 0:
                    # Get the first (and likely only) result
                    pred_boxes = results[0]["boxes"]
                    pred_labels = results[0]["labels"]
                    pred_scores = results[0]["scores"]
                    
                    with open(output_file, 'w') as f:
                        # Write coordinates to file in the required YOLO format: class_idx x_center y_center width height
                        for i, (box, label, score) in enumerate(zip(pred_boxes, pred_labels, pred_scores), 1):
                            # Convert box coordinates to integers
                            box = [int(b) for b in box]

                            # Normalize coordinates for YOLO format
                            x_center = (box[0] + box[2]) / 2 / image_width
                            y_center = (box[1] + box[3]) / 2 / image_height
                            width = (box[2] - box[0]) / image_width
                            height = (box[3] - box[1]) / image_height
                            
                            # Write normalized coordinates to file
                            f.write(f"{class_idx} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

                else:
                    print(f"No objects detected in {image_name}.")

                # Save the image (without bounding boxes)
                output_image_path = os.path.join(images_dir, f"{os.path.splitext(image_name)[0]}.png")
                image.save(output_image_path)


--------------------------------------------------------
Processing class: Alagao (Class 0)
->Processing image: Alagao (1).jpg
->Processing image: Alagao (10).jpg
->Processing image: Alagao (100).jpg
->Processing image: Alagao (101).jpg
->Processing image: Alagao (102).jpg
->Processing image: Alagao (103).jpg
->Processing image: Alagao (104).jpg
->Processing image: Alagao (105).jpg
->Processing image: Alagao (106).jpg
->Processing image: Alagao (107).jpg
->Processing image: Alagao (108).jpg
->Processing image: Alagao (109).jpg
->Processing image: Alagao (11).jpg
->Processing image: Alagao (110).jpg
->Processing image: Alagao (111).jpg
->Processing image: Alagao (112).jpg
->Processing image: Alagao (113).jpg
->Processing image: Alagao (114).jpg
->Processing image: Alagao (115).jpg
->Processing image: Alagao (116).jpg
->Processing image: Alagao (117).jpg
->Processing image: Alagao (118).jpg
->Processing image: Alagao (119).jpg
->Processing image: Alagao (12).jpg
->Processing image: Alag